# Audio Segmentation Pipeline

This Colab provides a streamlined pipeline for extracting labeled audio segments from S3 using Label Studio annotations and exporting them to Google Cloud Storage (GCS).

### Core Functions
1. **Manifest Retrieval**: Fetches target file lists from S3 and cross-references them with Label Studio ground truth exports. Only annotated files are processed.
2. **Local Audio Caching**: Downloads source audio into a local cache. On subsequent runs, it skips S3 downloads if the file is already present locally.
3. **Ground Truth Slicing**: Uses `librosa` and `soundfile` to precisely extract audio segments as defined by the start/end timestamps in the Label Studio export.
4. **Automated Export**:
    * Uploads processed FLAC segments to GCS organized by `example_id`.
    * Generates and uploads a `batch_manifest.jsonl` containing metadata for all segments (GCS paths, offsets, and durations) to facilitate downstream ASR evaluation.

**Note:** This Colab reads/writes content in the `wd-asr-chirp-evaluation` bucket in GCP.





In [7]:
# @title Install dependencies
!pip install -q --upgrade \
    boto3 \
    loguru

In [8]:
# @title Imports and constants
import os
import json
import sys
from typing import Any
from pathlib import Path
from urllib.parse import urlparse

import boto3
from botocore.config import Config
from botocore import UNSIGNED

import librosa
import soundfile as sf
from loguru import logger

from google.colab import auth
from google.cloud import storage


# Cloud & storage
S3_BUCKET = "watchduty-radio-transcription-data"
AWS_MANIFEST_URL = f"https://s3.amazonaws.com/{S3_BUCKET}/labelstudio/manifest.txt"
GCP_PROJECT_ID = "automatic-hawk-481415-m9"
GCS_BUCKET = "wd-asr-chirp-evaluation"
GCS_OUTPUT_PREFIX = "segmented_audio"
LOCAL_BASE_PATH = "/content"
CACHE_DIR = f"/{LOCAL_BASE_PATH}/raw_audio"
SEGMENTS_DIR = f"/{LOCAL_BASE_PATH}/segments"
LABEL_STUDIO_EXPORT_BLOB = "playground_export.json"
BATCH_MANIFEST_FILENAME = "batch_manifest.jsonl"

# Audio signal processing specs
SAMPLE_RATE = 16000
CHANNELS = 1

# Initialize loguru
logger.remove()
logger.add(sys.stderr, format="<level>{level}</level>: {message}");

In [9]:
# @title Authentication and client initialization
auth.authenticate_user()
!gcloud config set project {GCP_PROJECT_ID} --quiet

# S3 Client (Unsigned for public bucket access)
s3_config = Config(signature_version=UNSIGNED)
s3_client = boto3.client("s3", config=s3_config)

# Initialize GCS Client
gcs_client = storage.Client(project=GCP_PROJECT_ID)

Updated property [core/project].


In [10]:
# @title Define helper functions and pipeline

def get_paths_from_aws_manifest(s3_url: str) -> list[str]:
    """Fetches and parses the AWS manifest file to extract audio file paths."""
    parts = s3_url.replace("https://", "").split("/")
    bucket_name, key = parts[1], "/".join(parts[2:])
    response = s3_client.get_object(Bucket=bucket_name, Key=key)
    content = response["Body"].read().decode("utf-8")
    return [
        urlparse(line).path.lstrip("/")
        for line in content.splitlines()
        if line.startswith("http")
    ]

def ensure_local_audio(s3_path: str) -> str:
    """Ensures the audio file from S3 is available locally in CACHE_DIR."""
    filename = Path(s3_path).name
    local_path = Path(CACHE_DIR) / filename
    if not local_path.exists():
        logger.info(f"Downloading {filename} from S3...")
        s3_client.download_file(S3_BUCKET, s3_path, local_path)
    return str(local_path)

def get_paths_from_labelstudio(json_data: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """Parses Label Studio JSON to extract audio paths and transcribed segments only."""
    processed_files = []

    for item in json_data:
        audio_path = item.get("data", {}).get("audio", "")
        if not audio_path:
            continue

        segments = []
        annotations = item.get("annotations", [])

        for ann in annotations:
            results = ann.get("result", [])

            texts_by_id = {}
            for res in results:
                if res.get("type") == "textarea":
                    text_list = res.get("value", {}).get("text", [])
                    if text_list:
                        texts_by_id[res.get("id")] = text_list[0].strip()

            for res in results:
                if res.get("type") in ["labels", "audioregion"]:
                    region_id = res.get("id")
                    val = res.get("value", {})

                    if region_id in texts_by_id and texts_by_id[region_id]:
                        if val.get("start") is not None and val.get("end") is not None:
                            segments.append({
                                "start": val["start"],
                                "end": val["end"],
                                "text": texts_by_id[region_id]
                            })

        if segments:
            processed_files.append({
                "audio_path": audio_path,
                "segments": segments
            })

    return processed_files

def cleanup_gcs_output() -> None:
    """Deletes existing blobs in the output directory for a clean run."""
    bucket = gcs_client.bucket(GCS_BUCKET)
    blobs = list(bucket.list_blobs(prefix=GCS_OUTPUT_PREFIX))
    if blobs:
        logger.info(f"Cleaning existing files from gs://{GCS_BUCKET}/{GCS_OUTPUT_PREFIX}...")
        bucket.delete_blobs(blobs)

def slice_and_upload_audio(labeled_data: list[dict[str, Any]], s3_paths_map: dict[str, str]) -> list[dict[str, Any]]:
    """Slices audio based on labels and uploads to GCS."""
    bucket = gcs_client.bucket(GCS_BUCKET)
    manifest_entries = []
    os.makedirs(SEGMENTS_DIR, exist_ok=True)

    for item in labeled_data:
        audio_path = item["audio_path"]
        segments = item["segments"]
        lab_file = Path(audio_path).name

        if lab_file not in s3_paths_map:
            logger.warning(f"Labeled file {lab_file} not found in AWS manifest.")
            continue

        local_src_path = ensure_local_audio(s3_paths_map[lab_file])

        logger.info(f"Processing segments for: {lab_file}")
        y, sr = librosa.load(local_src_path, sr=SAMPLE_RATE)
        example_id = Path(lab_file).stem

        for i, seg in enumerate(segments):
            start_s, end_s = seg["start"], seg["end"]
            y_slice = y[int(start_s * sr):int(end_s * sr)]

            seg_id = f"{i:03d}"
            filename = f"{example_id}__seg{seg_id}.flac"
            local_slice_path = Path(SEGMENTS_DIR) / filename
            sf.write(str(local_slice_path), y_slice, sr)

            blob_name = f"{GCS_OUTPUT_PREFIX}/{example_id}/{filename}"
            blob = bucket.blob(blob_name)
            blob.upload_from_filename(str(local_slice_path))

            manifest_entries.append({
                "audio_filepath": f"gs://{GCS_BUCKET}/{blob_name}",
                "example_id": example_id,
                "offset": start_s,
                "duration": end_s - start_s,
                "segment_id": seg_id,
                "text": seg.get("text", "")
            })
    return manifest_entries

def run_ground_truth_pipeline() -> None:
    """Orchestrates extraction, caching, and upload."""
    os.makedirs(CACHE_DIR, exist_ok=True)
    cleanup_gcs_output()

    # Download export file from GCS
    bucket = gcs_client.bucket(GCS_BUCKET)
    blob = bucket.blob(LABEL_STUDIO_EXPORT_BLOB)
    json_data = json.loads(blob.download_as_string())

    labels = get_paths_from_labelstudio(json_data)

    all_s3_keys = get_paths_from_aws_manifest(AWS_MANIFEST_URL)
    s3_paths_map = {Path(p).name: p for p in all_s3_keys}

    manifest = slice_and_upload_audio(labels, s3_paths_map)

    local_manifest = Path(LOCAL_BASE_PATH) / BATCH_MANIFEST_FILENAME
    with open(local_manifest, "w") as f:
        for entry in manifest:
            f.write(json.dumps(entry) + "\n")

    gcs_manifest_path = f"{GCS_OUTPUT_PREFIX}/{BATCH_MANIFEST_FILENAME}"
    gcs_client.bucket(GCS_BUCKET).blob(gcs_manifest_path).upload_from_filename(str(local_manifest))

    logger.info(f"Pipeline Complete. {len(manifest)} segments uploaded.")

In [11]:
# @title Create the ground truth segments and manifest file
run_ground_truth_pipeline()

INFO: Downloading 5d995a78-ca_grass_valley2F202508202FCDF_Cmd_2_20250820_15.mp3 from S3...
INFO: Processing segments for: 5d995a78-ca_grass_valley2F202508202FCDF_Cmd_2_20250820_15.mp3
INFO: Downloading 06936ce3-ca_vacaville2F202506022FSolano_Co_Disp_20250602_16.mp3 from S3...
INFO: Processing segments for: 06936ce3-ca_vacaville2F202506022FSolano_Co_Disp_20250602_16.mp3
INFO: Downloading cc8dfbdd-202601141225-371759-141.mp3 from S3...
INFO: Processing segments for: cc8dfbdd-202601141225-371759-141.mp3
INFO: Downloading 9f66242e-202601140834-936159-909.mp3 from S3...
INFO: Processing segments for: 9f66242e-202601140834-936159-909.mp3
INFO: Downloading adc69b77-AEU-DISP_2026-01-03_12-18-41.mp3 from S3...
INFO: Processing segments for: adc69b77-AEU-DISP_2026-01-03_12-18-41.mp3
INFO: Downloading 10a60e74-AEU-DISP-CITH_2026-01-03_10-22-39.mp3 from S3...
INFO: Processing segments for: 10a60e74-AEU-DISP-CITH_2026-01-03_10-22-39.mp3
INFO: Pipeline Complete. 35 segments uploaded.
